In [1]:
import pandas as pd
import numpy as np

from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, StackingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold
from sklearn.feature_selection import SelectFromModel
from sklearn.metrics import accuracy_score, classification_report

from xgboost import XGBClassifier

from tensorflow.keras.models import Sequential, Model
from tensorflow.keras.layers import Dense, Dropout, BatchNormalization, Input
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping


In [2]:
# ================= LOAD DATA =================
train_df = pd.read_csv("train_data.csv")
test_df = pd.read_csv("test_data.csv")


X_train = train_df.drop("Label", axis=1)
y_train = train_df["Label"]

X_test = test_df.drop("Label", axis=1)
y_test = test_df["Label"]

In [3]:
# ================= LABEL ENCODING =================
le = LabelEncoder()
y_train = le.fit_transform(y_train.astype(str))
y_test = le.transform(y_test.astype(str))

# ================= FEATURE SELECTION =================
selector = SelectFromModel(RandomForestClassifier(n_estimators=80, n_jobs=-1), threshold="median")
X_train = selector.fit_transform(X_train, y_train)
X_test = selector.transform(X_test)

# ================= SCALING =================
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)


In [4]:
# ================= AUTOENCODER =================
input_dim = X_train.shape[1]

inp = Input(shape=(input_dim,))
x = Dense(128, activation='relu')(inp)
x = BatchNormalization()(x)
x = Dropout(0.2)(x)

x = Dense(64, activation='relu')(x)
x = BatchNormalization()(x)

x = Dense(32, activation='relu')(x)
bottleneck = Dense(24, activation='relu')(x)

x = Dense(32, activation='relu')(bottleneck)
x = BatchNormalization()(x)

x = Dense(64, activation='relu')(x)
x = Dense(128, activation='relu')(x)

out = Dense(input_dim, activation='linear')(x)

autoencoder = Model(inp, out)
autoencoder.compile(optimizer=Adam(0.0008), loss='mse')

early = EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)

autoencoder.fit(
    X_train, X_train,
    epochs=10,
    batch_size=512,
    validation_split=0.2,
    callbacks=[early],
    verbose=1
)

Epoch 1/10
183/183 ━━━━━━━━━━━━━━━━━━━━ 7s 12ms/step - loss: 0.3476 - val_loss: 0.1749
Epoch 2/10
183/183 ━━━━━━━━━━━━━━━━━━━━ 2s 10ms/step - loss: 0.1375 - val_loss: 0.0350
Epoch 3/10
183/183 ━━━━━━━━━━━━━━━━━━━━ 3s 11ms/step - loss: 0.1023 - val_loss: 0.0381
Epoch 4/10
183/183 ━━━━━━━━━━━━━━━━━━━━ 2s 7ms/step - loss: 0.0791 - val_loss: 0.0251
Epoch 5/10
183/183 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step - loss: 0.0705 - val_loss: 0.0249
Epoch 6/10
183/183 ━━━━━━━━━━━━━━━━━━━━ 3s 8ms/step - loss: 0.0808 - val_loss: 0.0240
Epoch 7/10
183/183 ━━━━━━━━━━━━━━━━━━━━ 2s 7ms/step - loss: 0.0569 - val_loss: 0.0196
Epoch 8/10
183/183 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - loss: 0.0532 - val_loss: 0.0202
Epoch 9/10
183/183 ━━━━━━━━━━━━━━━━━━━━ 3s 9ms/step - loss: 0.0451 - val_loss: 0.0149
Epoch 10/10
183/183 ━━━━━━━━━━━━━━━━━━━━ 3s 12ms/step - loss: 0.0469 - val_loss: 0.0149


In [5]:
# Encoder
encoder = Model(inp, bottleneck)

X_train_ae = encoder.predict(X_train)
X_test_ae = encoder.predict(X_test)

# Reconstruction error
recon_train = autoencoder.predict(X_train)
mse_train = np.mean((X_train - recon_train)**2, axis=1)

recon_test = autoencoder.predict(X_test)
mse_test = np.mean((X_test - recon_test)**2, axis=1)

# Combine features
X_train_final = np.hstack((X_train, X_train_ae, mse_train.reshape(-1,1)))
X_test_final = np.hstack((X_test, X_test_ae, mse_test.reshape(-1,1)))


3653/3653 ━━━━━━━━━━━━━━━━━━━━ 5s 1ms/step
1217/1217 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step
3653/3653 ━━━━━━━━━━━━━━━━━━━━ 5s 1ms/step
1217/1217 ━━━━━━━━━━━━━━━━━━━━ 2s 1ms/step


In [6]:
# ================= ML STACKING =================
rf = RandomForestClassifier(
    n_estimators=120,
    max_depth=16,
    class_weight='balanced',
    n_jobs=-1
)

gb = GradientBoostingClassifier(
    n_estimators=100,
    learning_rate=0.08
)

xgb = XGBClassifier(
    n_estimators=120,
    max_depth=6,
    learning_rate=0.08,
    subsample=0.8,
    colsample_bytree=0.8,
    eval_metric='mlogloss',
    tree_method='hist',
    n_jobs=-1
)

base_models = [('rf', rf), ('gb', gb), ('xgb', xgb)]

meta = LogisticRegression(max_iter=400)

cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)

stack = StackingClassifier(
    estimators=base_models,
    final_estimator=meta,
    cv=cv,
    passthrough=True,
    n_jobs=-1
)

print("Training ML...")
stack.fit(X_train_final, y_train)


Training ML...


,estimators,"[('rf', ...), ('gb', ...), ...]"
,final_estimator,LogisticRegre...(max_iter=400)
,cv,StratifiedKFo... shuffle=True)
,stack_method,'auto'
,n_jobs,-1
,passthrough,True
,verbose,0
,n_estimators,120
,criterion,'gini'
,max_depth,16
,min_samples_split,2


In [7]:
# ================= DEEP LEARNING =================
dl = Sequential([
    Input(shape=(X_train_final.shape[1],)),
    Dense(256, activation='relu'),
    BatchNormalization(),
    Dropout(0.25),
    Dense(128, activation='relu'),
    BatchNormalization(),
    Dropout(0.25),
    Dense(64, activation='relu'),
    Dense(len(set(y_train)), activation='softmax')
])

dl.compile(
    optimizer=Adam(0.0008),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

dl.fit(
    X_train_final, y_train,
    epochs=10,
    batch_size=512,
    validation_split=0.2,
    callbacks=[early],
    verbose=1
)

# ================= PREDICTIONS =================
print("Predicting...")

ml_probs = stack.predict_proba(X_test_final)
ml_preds = np.argmax(ml_probs, axis=1)

dl_probs = dl.predict(X_test_final)
dl_preds = np.argmax(dl_probs, axis=1)

threshold = np.percentile(mse_test, 98)


Epoch 1/10
183/183 ━━━━━━━━━━━━━━━━━━━━ 5s 16ms/step - accuracy: 0.9449 - loss: 0.1947 - val_accuracy: 0.9536 - val_loss: 0.3864
Epoch 2/10
183/183 ━━━━━━━━━━━━━━━━━━━━ 6s 18ms/step - accuracy: 0.9823 - loss: 0.0640 - val_accuracy: 0.9670 - val_loss: 0.1330
Epoch 3/10
183/183 ━━━━━━━━━━━━━━━━━━━━ 5s 18ms/step - accuracy: 0.9839 - loss: 0.0550 - val_accuracy: 0.9674 - val_loss: 0.1028
Predicting...
1217/1217 ━━━━━━━━━━━━━━━━━━━━ 3s 2ms/step


In [8]:
# ================= PREDICTIONS =================
print("Predicting...")

ml_probs = stack.predict_proba(X_test_final)
ml_preds = np.argmax(ml_probs, axis=1)

dl_probs = dl.predict(X_test_final)
dl_preds = np.argmax(dl_probs, axis=1)

threshold = np.percentile(mse_test, 98)

Predicting...
1217/1217 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step


In [9]:
# ================= HYBRID LOGIC =================
final_preds = []

for i in range(len(X_test)):

    ml_conf = np.max(ml_probs[i])
    dl_conf = np.max(dl_probs[i])
    error = mse_test[i]

    if error > threshold * 1.5:
        final_preds.append(ml_preds[i])

    elif ml_conf > 0.80:
        final_preds.append(ml_preds[i])

    elif dl_conf > 0.75:
        final_preds.append(dl_preds[i])

    else:
        combined = (ml_probs[i]*0.7 + dl_probs[i]*0.3)
        final_preds.append(np.argmax(combined))

final_preds = np.array(final_preds)


In [10]:
# ================= RESULTS =================
print("\nFinal Accuracy:", accuracy_score(y_test, final_preds))
print("\nClassification Report:\n", classification_report(y_test, final_preds))


Final Accuracy: 0.754560402856996

Classification Report:
               precision    recall  f1-score   support

           0       0.99      1.00      1.00     10847
           1       0.85      0.97      0.91      1440
           2       0.93      0.97      0.95      6212
           3       1.00      0.68      0.81       598
           4       0.07      1.00      0.14       533
           5       0.83      0.98      0.90     10420
           6       1.00      0.00      0.00      8872

    accuracy                           0.75     38922
   macro avg       0.81      0.80      0.67     38922
weighted avg       0.92      0.75      0.72     38922

